# MangoLeaf AI V4.5.0: fit and evaluate rejection settings

This experimental notebook uses **one completed V4.5.0 training run**. That training run learns the
seven mango classes and uses separate Outlier Exposure (OE) photos to discourage confident guesses
on unsupported content. This notebook changes **rejection settings only**. It never retrains the model.

All new outputs stay in **My Drive / mangoleaf_experiments / v4.5.0 / threshold_fitting**.
The original mango, OE, and OOD photo folders remain in place. No V4.3.0 or V4.4.0 run, split,
weights, masks, or result files are required. The raw OOD folder name `mango_ood_test_v4_4_0`
identifies your existing collection; it does not import an old model or old thresholds.

Run each step in order. Complete the CSV photo reviews before Step 6. Mango test photos and OOD
reporting photos cannot be predicted until rejection settings are frozen in Step 8.
Existing study OOD photos have already been used in earlier experiments, so their results are
reused-study evidence. Appending fresh photos does not make the entire collection a pristine holdout.

**Status:** unexecuted workflow template. It contains no claimed training scores, no measured browser
parity, and no model adoption or deployment. If the fixed validation-retention requirement cannot be
met, this notebook records infeasibility and stops before reporting predictions or candidate export.


## Step 1: Mount your own Drive and choose a completed run

**What:** Connect your original Google account and set the new training folder name.

**Why:** Shared shortcuts and similarly named folders can point to a different place from the completed model.

**How:** First open Runtime > Change runtime type, keep Python 3 and a GPU such as T4, and choose Runtime Version 2026.04 (Python 3.12.13). If unavailable, choose a listed Python 3.11/3.12 runtime from the [official Colab runtime list](https://research.google.com/colaboratory/runtime-version-faq.html). Then use a completed V4.5.0 run under your own My Drive. Paste only its folder name, or its full absolute path. If OOD is shared, add its shortcut to My Drive and set ALLOW_SHARED_INPUT_SHORTCUTS = True. Read the printed selected and resolved paths before continuing.

**Expected result:** The completed training run and outputs use the mounted account's actual mangoleaf_experiments folder. OOD uses an accessible My Drive folder or explicitly enabled shortcut.


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import math
import random
import shutil
import subprocess
import sys
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

# Check the actual interpreter, not only the Python 3 / T4 UI labels.
print('Notebook Python:', sys.version.split()[0])
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        'Unsupported Python ' + sys.version.split()[0] + '. '
        'Open Runtime > Change runtime type, keep a GPU such as T4, and select '
        'Runtime Version 2026.04 (Python 3.12.13), then rerun Step 1. '
        'If unavailable, use a listed Python 3.11/3.12 runtime: '
        'https://research.google.com/colaboratory/runtime-version-faq.html. '
        'These TensorFlow 2.19.0 pins must not run on an unsupported Python version.'
    )

from google.colab import drive

drive.mount('/content/drive')

# EDIT THIS NAME after completing the matching V4.5.0 training notebook.
TRAINING_RUN_NAME = 'PASTE_COMPLETED_V4_5_0_TRAINING_FOLDER_NAME'
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.5.0')
OOD_ROOT = Path('/content/drive/MyDrive/mango_ood_test_v4_4_0')
# Enable only for the reviewed shared OOD folder's My Drive shortcut.
ALLOW_SHARED_INPUT_SHORTCUTS = False

# First run requires 500 photos: existing 200 plus 300 uploaded before training.
# Set all three counts to 200 / 100 / 100 for a different explicit protocol.
# Do not borrow OE training photos to increase the held-out OOD pool.
EXPECTED_OOD_IMAGES = 500
TARGET_TUNING_IMAGES = 250
TARGET_REPORTING_IMAGES = 250
SEED = 42
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.70
MINIMUM_VALIDATION_RETENTION = 0.95
SIMILARITY_DISTANCE = 5
GRID = [number / 100 for number in range(101)]
EXPERIMENT_VERSION = 'v4.5.0'
CLASS_NAMES = [
    'anthracnose', 'bacterial_black_spot', 'bacterial_canker',
    'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold',
]
IMAGE_SIZE = (224, 224)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}

def drive_input_path(path, allow_shared=False, my_drive=Path('/content/drive/MyDrive')):
    """Accept selected My Drive inputs, with explicit shared-shortcut permission."""
    selected = Path(path)
    my_drive = Path(my_drive)
    if (not selected.is_absolute() or '..' in selected.parts
            or selected == my_drive or not selected.is_relative_to(my_drive)):
        raise ValueError('Select an input folder through a path below My Drive: ' + str(selected))

    resolved = selected.resolve()
    owned_root = my_drive.resolve()
    shortcut_root = (my_drive.parent / '.shortcut-targets-by-id').resolve()
    owned = resolved != owned_root and resolved.is_relative_to(owned_root)
    shared = (resolved.is_relative_to(shortcut_root)
              and len(resolved.relative_to(shortcut_root).parts) >= 2)
    print('Input:', selected, '\nResolved:', resolved)
    if not owned and not (allow_shared and shared):
        raise ValueError(
            'Input resolves outside My Drive. For a reviewed shared input, add its shortcut '
            'to My Drive and set ALLOW_SHARED_INPUT_SHORTCUTS = True. '
            'Other external paths remain unsupported: ' + str(resolved)
        )
    if not resolved.is_dir():
        raise FileNotFoundError('Input folder missing or inaccessible in the mounted account: ' + str(selected))
    return selected


def owned_my_drive_path(path, my_drive=Path('/content/drive/MyDrive')):
    """Keep outputs and completed training runs in the mounted account's My Drive."""
    selected = Path(path)
    resolved = selected.resolve()
    owned_root = Path(my_drive).resolve()
    print('Selected location:', selected)
    print('Resolved location:', resolved)
    print('Exists:', resolved.exists())
    if resolved == owned_root or not resolved.is_relative_to(owned_root):
        raise ValueError(
            'Output and training-run paths must resolve inside this mounted account\'s '
            'My Drive. Select its actual mangoleaf_experiments folder, not a shortcut '
            'to another account\'s output folder: ' + str(resolved)
        )
    return resolved


EXPERIMENT_ROOT = owned_my_drive_path(EXPERIMENT_ROOT)
OOD_ROOT = drive_input_path(OOD_ROOT, allow_shared=ALLOW_SHARED_INPUT_SHORTCUTS)
training_argument = Path(TRAINING_RUN_NAME)
if TRAINING_RUN_NAME.startswith('PASTE_'):
    raise ValueError('Paste the completed V4.5.0 training folder name above first.')
TRAINING_RUN = owned_my_drive_path(
    training_argument if training_argument.is_absolute()
    else EXPERIMENT_ROOT / 'training' / training_argument
)
training_root = (EXPERIMENT_ROOT / 'training').resolve()
if TRAINING_RUN == training_root or training_root not in TRAINING_RUN.parents:
    raise ValueError('Select a completed run inside mangoleaf_experiments/v4.5.0/training.')
if not TRAINING_RUN.is_dir() or not OOD_ROOT.is_dir():
    raise FileNotFoundError('Check the printed training and OOD folders before continuing.')
print('Confidence stays fixed:', CONFIDENCE_THRESHOLD)
print('Minimum validation acceptance:', MINIMUM_VALIDATION_RETENTION)


### Normal Colab runtime

The next cell uses the same reviewed scientific and security package versions as the new V4.5.0 training template. It updates pip before the remaining installation; exact helper pins replace older vulnerable Colab packages. Failed audits print their package and fix details and remain blocking.
It does not install a preserved historical requirements snapshot and does not start a custom
notebook kernel. If packages change, restart the Colab session, then rerun Steps 1 and 2.
The Python 3 label and T4 GPU choice do not identify the Python minor version. The setup cell
prints it and requires Python 3.11/3.12 before installing TensorFlow 2.19.0.
This threshold workflow does not install or invoke TensorFlow.js conversion: it copies the
already sealed browser export produced by training.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

# Check Python before mounting Drive or installing scientific packages.
print('Notebook Python:', sys.version.split()[0])
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        'Unsupported Python ' + sys.version.split()[0] + '. '
        'Open Runtime > Change runtime type, keep a GPU such as T4, and select '
        'Runtime Version 2026.04 (Python 3.12.13), then rerun Step 1. '
        'If unavailable, use a listed Python 3.11/3.12 runtime: '
        'https://research.google.com/colaboratory/runtime-version-faq.html. '
        'These TensorFlow 2.19.0 pins must not run on an unsupported Python version.'
    )

from google.colab import drive

drive.mount('/content/drive')

TRAINING_PACKAGES = [
    'tensorflow==2.19.0',
    'keras==3.15.1',
    'numpy==2.1.3',
    'Pillow==12.3.0',
    'ImageHash==4.3.2',
    'scipy==1.16.3',
    'matplotlib==3.11.2',
    'scikit-learn==1.9.1',
    'pip-audit==2.10.1',
    'packaging==25.0',
    'uv==0.12.23',
    # Explicit security pins prevent pip from retaining vulnerable provider helpers.
    'pip==26.2.1',
    'setuptools==83.0.0',
    'idna==3.15',
    'msgpack==1.2.1',
    'requests==2.33.0',
    'urllib3==2.8.0',
    'werkzeug==3.1.9',
]
setup_directory = Path('/content/mangoleaf-v450-threshold-setup')
setup_directory.mkdir(exist_ok=True)
setup_receipt = setup_directory / 'installation.json'
installed = json.loads(setup_receipt.read_text()) if setup_receipt.is_file() else None

if installed is None:
    # Upgrade the installer first; this subprocess does not import scientific modules.
    pip_pin = next(pin for pin in TRAINING_PACKAGES if pin.startswith('pip=='))
    bootstrap = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', '--no-deps', pip_pin],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'pip-bootstrap.log').write_text(bootstrap.stdout + '\n' + bootstrap.stderr)
    if bootstrap.returncode != 0:
        raise RuntimeError('Installer update failed. Read: ' + str(setup_directory / 'pip-bootstrap.log'))
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', *TRAINING_PACKAGES],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'installation.log').write_text(result.stdout + '\n' + result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Package installation failed. Read: ' + str(setup_directory / 'installation.log'))
    setup_receipt.write_text(json.dumps({
        'packages': TRAINING_PACKAGES, 'installationPid': os.getpid(),
        'python': sys.version, 'restartRequired': True,
    }, indent=2) + '\n')
    print('Installation complete. Restart session, then rerun Step 1 and continue.')
else:
    if installed['packages'] != TRAINING_PACKAGES:
        raise RuntimeError('Setup pins changed. Use a fresh runtime rather than mixing installation attempts.')
    print('Packages already installed. Restart completed:', installed['installationPid'] != os.getpid())


if json.loads(setup_receipt.read_text())['installationPid'] == os.getpid():
    raise RuntimeError('Restart the Colab session after installation, then rerun from Step 1.')


import csv
import hashlib
import importlib.metadata
import math
import random
import shutil
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import PurePosixPath

import imagehash
import matplotlib.pyplot as plt
import numpy as np
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from PIL import Image, ImageDraw, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

for pin in TRAINING_PACKAGES:
    name, expected = pin.split('==')
    if importlib.metadata.version(name) != expected:
        raise RuntimeError('Runtime version differs from chosen pin: ' + pin)

# Audit only the transitive research runtime, rather than treating every
# unrelated preinstalled Colab provider package as an executed experiment dependency.
def installed_research_dependencies(pins):
    names = set()
    examined_contexts = set()
    pending_requirements = [Requirement(pin) for pin in pins]
    while pending_requirements:
        requirement = pending_requirements.pop()
        name = canonicalize_name(requirement.name)
        distribution = importlib.metadata.distribution(name)
        if not requirement.specifier.contains(distribution.version, prereleases=True):
            raise RuntimeError(
                'Research dependency conflict: ' + str(requirement)
                + '; installed ' + distribution.version
            )
        names.add(name)
        # Extras such as CacheControl[filecache] add dependencies that must be audited.
        # Revisit a package when a later requirement enables a different extra.
        for extra in {''} | requirement.extras:
            context = (name, extra)
            if context in examined_contexts:
                continue
            examined_contexts.add(context)
            for dependency in distribution.requires or []:
                child = Requirement(dependency)
                if child.marker is None or child.marker.evaluate({'extra': extra}):
                    pending_requirements.append(child)
    return names


research_names = installed_research_dependencies(TRAINING_PACKAGES)

runtime_lock = setup_directory / 'training-runtime-lock.txt'
runtime_lock.write_text(''.join(
    name + '==' + importlib.metadata.version(name) + '\n'
    for name in sorted(research_names)
), encoding='utf-8')
audit_path = setup_directory / 'training-audit.json'
audit = subprocess.run([
    sys.executable, '-m', 'pip_audit', '-r', str(runtime_lock),
    '--no-deps', '--disable-pip', '--format=json', '--output', str(audit_path),
    '--progress-spinner=off',
], capture_output=True, text=True, check=False)
(setup_directory / 'training-audit.log').write_text(audit.stdout + '\n' + audit.stderr)
audit_report = json.loads(audit_path.read_text()) if audit_path.is_file() else {}
dependencies = audit_report.get('dependencies', [])
findings = [row for row in dependencies if row.get('vulns')]
skipped = [row for row in dependencies if row.get('skip_reason')]
audited_names = {canonicalize_name(row['name']) for row in dependencies if row.get('name')}
if audit.returncode != 0 or findings or skipped or audited_names != research_names:
    print('Dependency audit exit:', audit.returncode)
    for row in findings:
        print('Unresolved package:', row['name'], row.get('version', 'unknown'))
        details = sorted({
            vulnerability.get('id', 'unknown') + ' (fix versions: '
            + ', '.join(vulnerability.get('fix_versions', [])) + ')'
            for vulnerability in row['vulns']
        })
        for detail in details:
            print('  ', detail)
    for row in skipped:
        print('Skipped package:', row.get('name', 'unknown'), row['skip_reason'])
    print('Missing packages:', sorted(research_names - audited_names))
    print('Unexpected packages:', sorted(audited_names - research_names))
    print('Full audit and log:', audit_path, setup_directory / 'training-audit.log')
    raise RuntimeError('Threshold dependency audit incomplete or unresolved. Read ' + str(audit_path))

provider_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, check=False,
)
(setup_directory / 'provider-pip-check.log').write_text(provider_check.stdout + '\n' + provider_check.stderr)
if provider_check.returncode:
    print('Colab provider compatibility findings remain visible:', setup_directory / 'provider-pip-check.log')



from IPython.display import Markdown, display

if tf.__version__ != importlib.metadata.version('tensorflow'):
    raise RuntimeError('The imported TensorFlow differs from the installed package; restart the session.')
if tf.keras.__version__ != importlib.metadata.version('keras'):
    raise RuntimeError('Use the reviewed Keras runtime; do not enable a legacy Keras override.')

PINNED_PACKAGES = dict(pin.split('==') for pin in TRAINING_PACKAGES)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Threshold runtime ready:', PINNED_PACKAGES)


## Step 2: Define safe file and evidence helpers

**What:** Prepare fingerprint, relative-path, CSV, and immutable-record helpers.

**Why:** A review is valid only for the exact photo and model bytes that were reviewed.

**How:** Allow normal relative paths inside the chosen folder. Write evidence once; verify bytes and SHA-256 hashes before prediction.

**Expected result:** Unsafe ZIP paths or changed sealed files stop the run with a specific filename.


In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def inside(root, relative):
    # Only ordinary relative POSIX paths are accepted from a manifest or ZIP.
    name = PurePosixPath(str(relative))
    if name.is_absolute() or '..' in name.parts or '\\' in str(relative) or ':' in str(relative):
        raise ValueError('Unsafe relative path: ' + str(relative))
    result = (Path(root) / Path(*name.parts)).resolve()
    if Path(root).resolve() not in result.parents:
        raise ValueError('A manifest path escaped its expected folder.')
    return result


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_json_once(path, data):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(data, handle, indent=2, allow_nan=False)
        handle.write('\n')


def write_csv_once(path, rows, fields=None):
    rows = list(rows)
    fields = fields or list(rows[0])
    with Path(path).open('x', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        return list(csv.DictReader(handle))


def fingerprint_image(path, relative):
    with Image.open(path) as picture:
        rgb = ImageOps.exif_transpose(picture).convert('RGB')
        rgb.load()
        decoded = hashlib.sha256(str(rgb.size).encode('utf-8') + rgb.tobytes()).hexdigest()
        perceptual = str(imagehash.phash(rgb, hash_size=8))
        dimensions = list(rgb.size)
    fingerprint = sha256_file(path)
    return {
        'id': fingerprint[:20], 'relativePath': relative,
        'sha256': fingerprint, 'bytes': Path(path).stat().st_size,
        'decodedRgbSha256': decoded, 'phash': perceptual,
        'dimensions': dimensions,
    }


def perceptual_distance(first, second):
    return (int(first, 16) ^ int(second, 16)).bit_count()


def package_versions():
    versions = {}
    for name in ['tensorflow', 'keras', 'numpy', 'Pillow', 'ImageHash', 'scipy', 'matplotlib']:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def verify_artifacts():
    if sha256_file(COMPLETE_PATH) != COMPLETE_HASH:
        raise RuntimeError('The training completion record changed.')
    for relative, expected in complete['artifacts'].items():
        artifact = inside(TRAINING_RUN, relative)
        if not artifact.is_file() or artifact.stat().st_size != expected['bytes']:
            raise RuntimeError('A completed training artifact is missing or changed: ' + relative)
        if sha256_file(artifact) != expected['sha256']:
            raise RuntimeError('A completed training artifact fingerprint changed: ' + relative)


def verify_photo(record, root):
    path = inside(root, record['relativePath'])
    if path.stat().st_size != record['bytes'] or sha256_file(path) != record['sha256']:
        raise RuntimeError('A reviewed photo changed: ' + record['relativePath'])
    return path


## Step 3: Verify training completion and unpack its fresh split

**What:** Check all training receipts and copy the saved split to temporary Colab storage.

**Why:** Threshold fitting must use the exact calibrated model and the exact validation/test membership from this new training run.

**How:** Verify 15 completed epochs, seven classes, folded temperature, unused test predictions, inventory hashes, package versions, ZIP entries, and every extracted photo.

**Expected result:** A new threshold run folder is created under mangoleaf_experiments/v4.5.0/threshold_fitting. Mango test images remain unpredicted.


In [ ]:
COMPLETE_PATH = TRAINING_RUN / 'training_complete.json'
if not COMPLETE_PATH.is_file():
    raise FileNotFoundError('Finish the fresh V4.5.0 training notebook and paste its run folder in Step 1.')

complete = read_json(COMPLETE_PATH)
COMPLETE_HASH = sha256_file(COMPLETE_PATH)
if complete.get('experimentVersion') != EXPERIMENT_VERSION:
    raise ValueError('The input is not a completed V4.5.0 experiment.')
if complete.get('completedEpochs') != 15:
    raise ValueError('The fresh run must finish all 5 + 10 epochs before threshold fitting.')
if complete.get('classNames') != CLASS_NAMES:
    raise ValueError('The training class order does not match the agreed seven classes.')
if complete.get('foldedTemperature') is not True:
    raise ValueError('The completed model must already include its fitted temperature.')
if complete.get('testPredictionsRun') is not False:
    raise ValueError('Mango test predictions must remain unused until rejection settings are frozen.')

required_artifacts = {
    'input_manifest.json', 'mango_inventory.json', 'oe_inventory.json', 'oe_training_manifest.json', 'oe_dataset_manifest.json', 'held_out_ood_inventory.json',
    'dataset_split.zip', 'model_calibrated.keras', 'class_names.json',
    'temperature.json', 'runtime.json', 'plan.json',
    'validation_probabilities.npy', 'validation_logits.npy', 'frozen_inputs.json', 'tfjs_model/model.json', 'tfjs_model/model_config.json',
}
if not required_artifacts.issubset(complete.get('artifacts', {})):
    raise ValueError('The completed training record lacks required fresh artifacts.')
verify_artifacts()
temperature_record = read_json(TRAINING_RUN / 'temperature.json')
if (temperature_record.get('temperatureFoldedIntoWeights') is not True
        or temperature_record.get('foldCount') != 1
        or temperature_record.get('fitSubset') != 'val'
        or temperature_record.get('fitIncludesOE') is not False
        or temperature_record.get('fitIncludesOOD') is not False):
    raise ValueError('Temperature must be fitted on clean mango validation only and folded exactly once.')
if not math.isclose(float(temperature_record['temperature']),
                    float(complete['calibratedTemperature']), rel_tol=1e-12):
    raise ValueError('The temperature receipt differs from the completed model.')
if EXPECTED_OOD_IMAGES is not None and complete.get('heldOutOodCount') != EXPECTED_OOD_IMAGES:
    raise ValueError('The fresh training run used a different held-out OOD protocol count.')

manifest = read_json(TRAINING_RUN / 'input_manifest.json')
if sha256_file(TRAINING_RUN / 'input_manifest.json') != complete['inputManifestSha256']:
    raise ValueError('The completed split fingerprint does not match its manifest.')
if read_json(TRAINING_RUN / 'class_names.json') != CLASS_NAMES:
    raise ValueError('The saved model class list differs.')

saved_runtime = read_json(TRAINING_RUN / 'runtime.json')
current_versions = package_versions()
for package in ['tensorflow', 'keras']:
    expected = saved_runtime['packages'][package]
    if not expected or current_versions[package] != expected:
        raise RuntimeError('Use the same reviewed ' + package + ' version as the completed training run: ' + str(expected))

run_id = 'v4-5-0-thresholds-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = EXPERIMENT_ROOT / 'threshold_fitting' / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
SCRATCH_DIR = Path('/content') / run_id
SPLIT_DIR = SCRATCH_DIR / 'dataset_split'
SPLIT_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_DIR = RUN_DIR / 'figures'
FIGURE_DIR.mkdir()
shutil.copytree(setup_directory, RUN_DIR / 'runtime_preflight')

mango_records = manifest['images']
if set(record['subset'] for record in mango_records) != {'train', 'val', 'test'}:
    raise ValueError('The fresh split must contain train, val, and test.')
expected_paths = set()
for record in mango_records:
    name = PurePosixPath(record['relativePath'])
    if record['label'] not in CLASS_NAMES or name.parts[:2] != (record['subset'], record['label']):
        raise ValueError('A mango record has an unexpected class or subset path.')
    inside(SPLIT_DIR, record['relativePath'])
    expected_paths.add(record['relativePath'])
if len(expected_paths) != len(mango_records):
    raise ValueError('The split contains repeated paths.')

with zipfile.ZipFile(TRAINING_RUN / 'dataset_split.zip') as archive:
    members = [member for member in archive.infolist() if not member.is_dir()]
    archive_names = [member.filename for member in members]
    if len(set(archive_names)) != len(archive_names) or set(archive_names) != expected_paths:
        raise ValueError('The ZIP image identities differ from the completed split.')
    expected_by_path = {record['relativePath']: record for record in mango_records}
    for member in members:
        destination = inside(SPLIT_DIR, member.filename)
        if member.file_size != expected_by_path[member.filename]['bytes']:
            raise ValueError('An archive member has an unexpected byte count.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open('xb') as target:
            shutil.copyfileobj(source, target)

for record in mango_records:
    verify_photo(record, SPLIT_DIR)

validation_records = [record for record in mango_records if record['subset'] == 'val']
test_records = [record for record in mango_records if record['subset'] == 'test']
if any(not any(record['label'] == label for record in validation_records) for label in CLASS_NAMES):
    raise ValueError('Every supported class needs validation examples.')

write_json_once(RUN_DIR / 'runtime.json', {
    'distributions': current_versions, 'pythonVersion': __import__('sys').version,
    'trainingRuntimeSha256': sha256_file(TRAINING_RUN / 'runtime.json'),
    'runtimePins': PINNED_PACKAGES,
    'createdAt': utc_now(),
})
print('Fresh run output:', RUN_DIR)
print('Saved split counts:', dict(Counter(record['subset'] for record in mango_records)))
print('No mango test or OOD reporting predictions have run.')


# Related mango groups must be wholly within one saved subset.
mango_subsets_by_group = defaultdict(set)
for record in mango_records:
    mango_subsets_by_group[record['groupId']].add(record['subset'])
if any(len(subsets) != 1 for subsets in mango_subsets_by_group.values()):
    raise ValueError('A related mango group crosses the saved train/validation/test split.')
write_json_once(RUN_DIR / 'training_reference.json', {
    'trainingRun': str(TRAINING_RUN), 'trainingCompleteSha256': COMPLETE_HASH,
    'configurationHash': complete['configurationHash'],
    'oeInventorySha256': sha256_file(TRAINING_RUN / 'oe_inventory.json'),
    'heldOutOodInventorySha256': sha256_file(TRAINING_RUN / 'held_out_ood_inventory.json'),
    'classNames': CLASS_NAMES, 'createdAt': utc_now(),
})


## Step 4: Count OOD photos and check overlap with mango and OE

**What:** Inventory the unsupported photo pool and flag exact, decoded, and visually similar matches.

**Why:** OE is training data. An OE photo, or a related capture, cannot also supply independent OOD tuning or reporting evidence.

**How:** Require the same 500-photo pool that was frozen before training, and compare every photo in the frozen pool with the frozen raw mango and approved OE inventories.

**Expected result:** You see the actual OOD count, the number frozen before training, and review CSVs. No extra images are invented or copied from OE.


In [ ]:
def inventory_ood_folder(root):
    photos, unreadable, other_files = [], [], []
    for path in sorted(root.rglob('*')):
        if not path.is_file():
            continue
        relative = path.relative_to(root).as_posix()
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            other_files.append(relative)
            continue
        try:
            record = fingerprint_image(path, relative)
            # Path-specific IDs let exact copies be reviewed separately.
            record['id'] = hashlib.sha256((relative + record['sha256']).encode()).hexdigest()[:20]
            photos.append(record)
        except Exception as error:
            unreadable.append({'relativePath': relative, 'error': str(error)})
    if unreadable:
        raise ValueError('Correct unreadable OOD photos before continuing: ' + str(unreadable))
    if len({row['id'] for row in photos}) != len(photos):
        raise ValueError('A truncated OOD ID collision occurred; use full SHA-256 path IDs for a new run.')
    return photos, other_files

ood_records, other_files = inventory_ood_folder(OOD_ROOT)
if len(ood_records) < 2:
    raise ValueError('At least two reviewed OOD photos are needed for separate subsets.')
if EXPECTED_OOD_IMAGES is not None and len(ood_records) != EXPECTED_OOD_IMAGES:
    raise ValueError(f'Expected {EXPECTED_OOD_IMAGES} OOD photos; found {len(ood_records)}.')

original_ood = read_json(TRAINING_RUN / 'held_out_ood_inventory.json')['images']
current_by_path = {record['relativePath']: record for record in ood_records}
original_paths = set()
for record in original_ood:
    relative = record.get('sourceRelativePath', record.get('relativePath'))
    original_paths.add(relative)
    current = current_by_path.get(relative)
    if current is None or current['sha256'] != record['sha256']:
        raise ValueError('A pretraining held-out OOD photo was removed or changed: ' + str(relative))
if set(current_by_path) != original_paths:
    raise ValueError('OOD membership changed after training. Audit all 500 photos before a fresh training run.')
original_by_path = {
    row.get('sourceRelativePath', row.get('relativePath')): row for row in original_ood
}
for record in ood_records:
    original = original_by_path[record['relativePath']]
    record['groupIdBeforeThreshold'] = original.get('groupId', original['id'])
    record['captureGroup'] = original.get('captureGroup', '')
    record['sourceGroup'] = original.get('sourceGroup', '')

raw_mango_inventory = read_json(TRAINING_RUN / 'mango_inventory.json')['images']
# The raw inventory precedes acquisition review. Overlay its retained representative's
# frozen capture/source metadata without changing any raw filename or photo fingerprint.
mango_by_decoded = {row['decodedRgbSha256']: row for row in mango_records}
for raw in raw_mango_inventory:
    representative = mango_by_decoded.get(raw['decodedRgbSha256'])
    if representative is None:
        raise ValueError('A raw mango photo has no representative in the frozen split.')
    for field in ['captureGroup', 'sourceGroup']:
        raw[field] = representative.get(field, '')
approved_oe_inventory = read_json(TRAINING_RUN / 'oe_training_manifest.json')['images']
oe_inventory_by_id = {row['id']: row for row in read_json(TRAINING_RUN / 'oe_inventory.json')['images']}
for row in approved_oe_inventory:
    if row['id'] not in oe_inventory_by_id or oe_inventory_by_id[row['id']]['sha256'] != row['sha256']:
        raise ValueError('The approved OE training manifest differs from its reviewed inventory.')
external_records = [
    {'kind': kind, **record}
    for kind, records in [('mango', raw_mango_inventory), ('oe', approved_oe_inventory)]
    for record in records
]
pair_flags = []

def flag_pair(first, second, kind):
    exact = first['sha256'] == second['sha256']
    decoded_match = first['decodedRgbSha256'] == second['decodedRgbSha256']
    distance = perceptual_distance(first['phash'], second['phash'])
    if not (exact or decoded_match or distance <= SIMILARITY_DISTANCE):
        return
    relative = second.get('sourceRelativePath', second.get('relativePath', ''))
    identity = second['id'] if kind == 'ood' else hashlib.sha256(
        (kind + relative + second['sha256']).encode()
    ).hexdigest()[:20]
    pair_flags.append({
        'firstId': first['id'], 'secondKind': kind, 'secondId': identity,
        'firstPath': first['relativePath'], 'secondPath': relative,
        'exactBytes': str(exact).lower(), 'decodedMatch': str(decoded_match).lower(),
        'phashDistance': distance, 'relationship': 'REVIEW', 'notes': '',
    })

for index, first in enumerate(ood_records):
    for second in ood_records[index + 1:]:
        flag_pair(first, second, 'ood')
    for second in external_records:
        flag_pair(first, second, second['kind'])

write_json_once(RUN_DIR / 'ood_inventory.json', {
    'sourceRoot': str(OOD_ROOT), 'images': ood_records, 'nonImageFiles': other_files,
    'readableCount': len(ood_records), 'pretrainingHeldOutCount': len(original_ood),
    'reuseStatus': 'Existing study OOD pool; use history must be recorded per photo. Not a pristine blind holdout.',
})
write_csv_once(RUN_DIR / 'ood_review.csv', [
    {'id': record['id'], 'relativePath': record['relativePath'], 'keep': 'REVIEW',
     'unsupported': 'REVIEW', 'captureGroup': record['captureGroup'],
     'sourceGroup': record['sourceGroup'], 'sourceNotes': '',
     'priorUse': 'REVIEW', 'notes': ''}
    for record in ood_records
])
write_csv_once(RUN_DIR / 'similarity_review.csv', pair_flags, fields=[
    'firstId', 'secondKind', 'secondId', 'firstPath', 'secondPath',
    'exactBytes', 'decodedMatch', 'phashDistance', 'relationship', 'notes',
])
print('Actual readable OOD photos:', len(ood_records))
print('Same pretraining held-out membership:', len(original_paths))
print('Flagged pairs requiring review:', len(pair_flags))
print('Review CSV folder:', RUN_DIR)


## Step 5: Review unsupported scope, sources, and related captures

**What:** Show each OOD photo and complete the review CSVs without looking at model guesses.

**Why:** A supported mango photo is not OOD because its prediction is difficult. Related captures and previously reused photos must be identified honestly.

**How:** Use the rules below, export edited tables as CSV, and replace the matching CSV files in this run folder. Rerun only this preview step if needed.

**Expected result:** Every retained photo has yes/no decisions and honest source/use notes. Every flagged pair has a relationship and explanation.


### CSV review rules

| Column | What to enter |
| --- | --- |
| `keep` | `yes` or `no`. Keep only usable, correctly scoped unsupported photos. |
| `unsupported` | `yes` if the photo is outside all seven supported mango classes; otherwise `no`. |
| `captureGroup` | The same stable label for photos of the same object/burst/session that should remain together. Use the same label if that capture also exists in OE. Leave blank only for an independent capture. |
| `sourceGroup` | The same stable label for variants from one original image, video, or rotating-object sequence. Use the same label as OE when they share that source. Do not use a broad label such as `internet` to group unrelated photos. |
| `sourceNotes` | Honest source, capture, or URL notes for retained photos. If a precise public source is unknown, say so. Do not invent provenance. |
| `priorUse` | `previously_evaluated`, `new_for_v4_5_0`, or `unknown`, with the reason/history in `notes`. Earlier v4.3.0/v4.4.0 OOD study photos should be `previously_evaluated`. Newly appended does not automatically mean previously unused. |
| `notes` | Explain previous use, exclusions, and uncertain sources. Every retained row needs a note. |

In `similarity_review.csv`, change **only** `relationship` and `notes`:

- `same_capture`: same object/source image/burst or related variants; keep that OOD group together.
- `unrelated`: visually similar but genuinely separate content; explain the distinction.
- Exact or decoded duplicates cannot be called unrelated. Keep one OOD copy; exclude any OOD content
  that overlaps mango training/validation/test material or approved OE training material.
- A header-only similarity CSV means no flagged pairs; leave it unchanged.

Google Sheets edits do not update the original Drive CSV automatically. Use **File → Download →
Comma-separated values (.csv)**, then replace this run's `ood_review.csv` or `similarity_review.csv`.
A separate green Google Sheets file is not the file this notebook reads. Keep IDs, filenames, hashes,
and review-table headers unchanged. Review source/capture overlap even when no pHash pair is flagged.

The first-run protocol requires **500 photos: existing 200 plus 300 added before training**.
Targets are **250 tuning / 250 reporting**. If exclusions leave fewer than 500 photos, stop and
assemble/re-audit a valid pool before a fresh training run. Changing all three settings to
200 / 100 / 100 defines a different explicit protocol. No photos are downloaded or invented.


In [ ]:
def show_contact_sheets(records, root, destination, per_page=25):
    destination.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(records), per_page):
        page = records[start:start + per_page]
        canvas = Image.new('RGB', (1250, 1100), 'white')
        draw = ImageDraw.Draw(canvas)
        for offset, record in enumerate(page):
            left = (offset % 5) * 250
            top = (offset // 5) * 220
            with Image.open(inside(root, record['relativePath'])) as picture:
                thumbnail = ImageOps.exif_transpose(picture).convert('RGB')
                thumbnail.thumbnail((238, 174))
                canvas.paste(thumbnail, (left + 6, top + 4))
            draw.text((left + 6, top + 180), record['id'], fill='black')
            draw.text((left + 6, top + 197), Path(record['relativePath']).name[:34], fill='black')
        output = destination / ('ood_' + str(start // per_page + 1).zfill(2) + '.jpg')
        canvas.save(output, quality=92)
        display(canvas)


show_contact_sheets(ood_records, OOD_ROOT, RUN_DIR / 'review_contact_sheets')


def external_preview_path(pair):
    # The training run preserves the exact mango representatives and the approved OE originals.
    if pair['secondKind'] == 'ood':
        return inside(OOD_ROOT, pair['secondPath'])
    candidates = raw_mango_inventory if pair['secondKind'] == 'mango' else approved_oe_inventory
    original = next(row for row in candidates if row['sourceRelativePath'] == pair['secondPath'])
    if pair['secondKind'] == 'oe':
        copied = next(row for row in read_json(TRAINING_RUN / 'oe_dataset_manifest.json')['images']
                      if row['id'] == original['id'])
        return verify_photo(copied, TRAINING_RUN)
    representative = next(row for row in mango_records
                          if row['sha256'] == original['sha256']
                          or row['decodedRgbSha256'] == original['decodedRgbSha256'])
    return verify_photo(representative, SPLIT_DIR)


def show_pair_contact_sheets(pairs, per_page=10):
    destination = RUN_DIR / 'similarity_contact_sheets'
    destination.mkdir(exist_ok=True)
    for start in range(0, len(pairs), per_page):
        chosen = pairs[start:start + per_page]
        canvas = Image.new('RGB', (1200, len(chosen) * 225), 'white')
        draw = ImageDraw.Draw(canvas)
        for row_index, pair in enumerate(chosen):
            for column, photo_path, label in [
                (0, inside(OOD_ROOT, pair['firstPath']), 'OOD ' + pair['firstId']),
                (1, external_preview_path(pair), pair['secondKind'] + ' ' + pair['secondId']),
            ]:
                with Image.open(photo_path) as picture:
                    preview = ImageOps.exif_transpose(picture).convert('RGB')
                    preview.thumbnail((575, 185))
                    left, top = column * 600, row_index * 225
                    canvas.paste(preview, (left + 8, top + 5))
                draw.text((left + 8, top + 194), label, fill='black')
                draw.text((left + 8, top + 209),
                          'pHash distance=' + str(pair['phashDistance']) + ' | ' + photo_path.name[:60],
                          fill='black')
        canvas.save(destination / f'pairs_{start // per_page + 1:03d}.jpg', quality=92)
        display(canvas)


show_pair_contact_sheets(pair_flags)
print('Complete the review tables before continuing. No model predictions were used for this review.')


## Step 6: Freeze balanced whole-group tuning and reporting assignments

**What:** Validate decisions and split retained OOD groups into two equal sets.

**Why:** Related photos in both sets would make reporting look more independent than it is. Source/capture overlap with OE would leak training content into evaluation.

**How:** Join related OOD photos, block OE/mango overlaps, and use a seeded subset-sum assignment that never cuts a group in half. Targets are explicitly fixed at 250 photos per subset.

**Expected result:** The first run requires 500 retained photos: 250 tuning and 250 reporting, if whole groups permit those exact counts. Otherwise the run stops with group sizes.


In [ ]:
def assign_whole_groups(groups, target, seed):
    # A subset-sum search finds an exact count while keeping every group intact.
    if not isinstance(target, int) or isinstance(target, bool) or target < 1:
        raise ValueError('The tuning target must be a positive integer.')
    if not groups or any(not rows for rows in groups.values()):
        raise ValueError('Whole-photo groups must be nonempty.')
    ordered = sorted(groups)
    random.Random(seed).shuffle(ordered)
    reachable = {0: []}
    for group in ordered:
        size = len(groups[group])
        additions = {}
        for count, chosen in list(reachable.items()):
            new_count = count + size
            if new_count <= target and new_count not in reachable and new_count not in additions:
                additions[new_count] = chosen + [group]
        reachable.update(additions)
    if target not in reachable:
        raise ValueError('Cannot assign exactly ' + str(target) + ' tuning images without splitting groups. Group sizes: ' + str(sorted(len(rows) for rows in groups.values())))
    return set(reachable[target])


photo_review = read_csv(RUN_DIR / 'ood_review.csv')
similarity_review = read_csv(RUN_DIR / 'similarity_review.csv')
reviews_by_id = {row['id']: row for row in photo_review}
if len(reviews_by_id) != len(photo_review) or set(reviews_by_id) != {row['id'] for row in ood_records}:
    raise ValueError('Review IDs must match the complete OOD inventory exactly.')
for record in ood_records:
    review = reviews_by_id[record['id']]
    if review['relativePath'] != record['relativePath']:
        raise ValueError('A reviewed filename was changed.')
    if review['keep'] not in {'yes', 'no'} or review['unsupported'] not in {'yes', 'no'}:
        raise ValueError('Finish yes/no decisions for: ' + record['relativePath'])
    if review['keep'] == 'yes' and (review['unsupported'] != 'yes' or not review['sourceNotes'].strip()):
        raise ValueError('Kept photos need confirmed unsupported scope and honest source notes: ' + record['relativePath'])
    if review.get('priorUse') not in {'previously_evaluated', 'new_for_v4_5_0', 'unknown'}:
        raise ValueError('Record honest priorUse: ' + record['relativePath'])
    if review['keep'] == 'yes' and not review['notes'].strip():
        raise ValueError('Retained photos need honest previous-use/source notes: ' + record['relativePath'])

retained = [record for record in ood_records if reviews_by_id[record['id']]['keep'] == 'yes']
retained_ids = {record['id'] for record in retained}
parent = {record['id']: record['id'] for record in retained}


def find_group(identity):
    while parent[identity] != identity:
        parent[identity] = parent[parent[identity]]
        identity = parent[identity]
    return identity


def join_group(first, second):
    parent[find_group(second)] = find_group(first)


manual_groups = defaultdict(list)
for record in retained:
    for field in ['captureGroup', 'sourceGroup']:
        label = reviews_by_id[record['id']][field].strip()
        if label:
            manual_groups[(field, label)].append(record['id'])
for record in retained:
    manual_groups[('pretrainingGroup', record['groupIdBeforeThreshold'])].append(record['id'])
    # Preserve known acquisition relationships from the training audit even if a CSV label changes.
    for field in ['captureGroup', 'sourceGroup']:
        original_group = record[field].strip()
        if original_group:
            manual_groups[('pretraining_' + field, original_group)].append(record['id'])
for identities in manual_groups.values():
    for identity in identities[1:]:
        join_group(identities[0], identity)

expected_pairs = {(row['firstId'], row['secondKind'], row['secondId']): row for row in pair_flags}
received_pairs = {(row['firstId'], row['secondKind'], row['secondId']): row for row in similarity_review}
if len(received_pairs) != len(similarity_review) or set(received_pairs) != set(expected_pairs):
    raise ValueError('The similarity review must include every original flagged pair exactly once.')
for key, pair in received_pairs.items():
    original = expected_pairs[key]
    if any(str(pair[field]) != str(original[field]) for field in ['firstPath', 'secondPath', 'exactBytes', 'decodedMatch', 'phashDistance']):
        raise ValueError('A flagged pair identity was edited. Only relationship and notes may change.')
    if pair['relationship'] not in {'same_capture', 'unrelated'} or not pair['notes'].strip():
        raise ValueError('Finish the similarity decision and explanation for: ' + pair['firstPath'])
    first_kept = pair['firstId'] in retained_ids
    second_kept = pair['secondKind'] == 'ood' and pair['secondId'] in retained_ids
    identical = pair['exactBytes'] == 'true' or pair['decodedMatch'] == 'true'
    if first_kept and (second_kept or pair['secondKind'] != 'ood') and identical:
        raise ValueError('Exclude duplicate/overlapping content before splitting: ' + pair['firstPath'])
    if first_kept and pair['secondKind'] != 'ood' and pair['relationship'] == 'same_capture':
        raise ValueError('An OOD capture overlaps mango/OE material. Exclude: ' + pair['firstPath'])
    if first_kept and second_kept and pair['relationship'] == 'same_capture':
        join_group(pair['firstId'], pair['secondId'])

# Compare manual source/capture labels with frozen training inventories, even without a pHash hit.
for record in retained:
    review = reviews_by_id[record['id']]
    for external in external_records:
        for field in ['captureGroup', 'sourceGroup']:
            group = review[field].strip()
            if group and group == str(external.get(field, '')).strip():
                raise ValueError('OOD ' + field + ' overlaps ' + external['kind'] + ': ' + record['relativePath'])
if not retained or len(retained) % 2:
    raise ValueError('The retained pool must be positive and even for two equal subsets. Resolve review exclusions first.')
if (TARGET_TUNING_IMAGES is None) != (TARGET_REPORTING_IMAGES is None):
    raise ValueError('Leave both target counts as None, or set both to matching explicit counts.')
if TARGET_TUNING_IMAGES is None:
    TARGET_TUNING_IMAGES = len(retained) // 2
    TARGET_REPORTING_IMAGES = len(retained) // 2
if TARGET_TUNING_IMAGES != TARGET_REPORTING_IMAGES or TARGET_TUNING_IMAGES < 1:
    raise ValueError('Tuning and reporting targets must be equal positive integers.')
if len(retained) != TARGET_TUNING_IMAGES + TARGET_REPORTING_IMAGES:
    raise ValueError('Reviewed OOD pool contains ' + str(len(retained)) + ' photos, but the explicit targets total ' + str(TARGET_TUNING_IMAGES + TARGET_REPORTING_IMAGES) + '. Resolve exclusions or revise the targets before freezing.')

groups = defaultdict(list)
for record in retained:
    groups[find_group(record['id'])].append(record)
tuning_groups = assign_whole_groups(groups, TARGET_TUNING_IMAGES, SEED)
assigned_records = []
for identity, group in sorted(groups.items()):
    subset = 'ood_tune' if identity in tuning_groups else 'ood_report'
    for record in group:
        assigned_records.append({**record, 'groupId': identity, 'subset': subset,
            **{field: reviews_by_id[record['id']][field] for field in ['captureGroup', 'sourceGroup', 'sourceNotes', 'priorUse', 'notes']}})

assignment_counts = Counter(record['subset'] for record in assigned_records)
if assignment_counts != Counter({'ood_tune': TARGET_TUNING_IMAGES, 'ood_report': TARGET_REPORTING_IMAGES}):
    raise ValueError('The final assignments do not match the explicit targets.')
for record in ood_records:
    verify_photo(record, OOD_ROOT)
verify_artifacts()

evaluation_inputs = {
    'experimentVersion': EXPERIMENT_VERSION, 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_HASH, 'seed': SEED,
    'oodRoot': str(OOD_ROOT), 'oodInventorySha256': sha256_file(RUN_DIR / 'ood_inventory.json'),
    'photoReviewSha256': sha256_file(RUN_DIR / 'ood_review.csv'),
    'similarityReviewSha256': sha256_file(RUN_DIR / 'similarity_review.csv'),
    'oodImages': assigned_records, 'counts': dict(assignment_counts),
    'mangoInputManifestSha256': complete['inputManifestSha256'],
    'reuseStatus': 'Existing study pool plus newly collected photos; histories recorded. Not pristine external evidence.',
    'priorUseCounts': dict(Counter(row['priorUse'] for row in assigned_records)),
    'oeInventorySha256': sha256_file(TRAINING_RUN / 'oe_inventory.json'),
    'originalHeldOutInventorySha256': sha256_file(TRAINING_RUN / 'held_out_ood_inventory.json'),
    'expectedOodImages': EXPECTED_OOD_IMAGES,
    'oodRawCount': len(ood_records),
    'groupAudit': 'Whole capture/source/near-duplicate groups; exact/decoded training overlap excluded.',
    'frozenAt': utc_now(),
}
write_json_once(RUN_DIR / 'evaluation_inputs.json', evaluation_inputs)
INPUT_HASH = sha256_file(RUN_DIR / 'evaluation_inputs.json')
write_csv_once(RUN_DIR / 'ood_assignments.csv', [
    {'id': record['id'], 'relativePath': record['relativePath'], 'subset': record['subset'],
     'groupId': record['groupId'], 'sha256': record['sha256']}
    for record in assigned_records
])
write_json_once(RUN_DIR / 'input_lock.json', {'evaluationInputsSha256': INPUT_HASH, 'createdAt': utc_now()})
tuning_records = [record for record in assigned_records if record['subset'] == 'ood_tune']
reporting_records = [record for record in assigned_records if record['subset'] == 'ood_report']
print('Frozen OOD group counts:', dict(assignment_counts))
print('No tuning or reporting photo belongs to both groups.')


## Step 7: Define prediction, confidence, margin, and entropy

**What:** Prepare identical preprocessing and immutable prediction calculations.

**Why:** Different resizing, an extra 1/255 scale, or applying temperature twice would change the scores being evaluated.

**How:** Decode RGB with the same loader as training, resize bilinearly to 224 × 224 without antialiasing, keep float32 values in 0–255, and call the already calibrated model with training=False.

**Expected result:** Seven finite probabilities sum to one for every photo. Confidence is the largest score, margin is the top-two gap, and normalized entropy measures uncertainty from zero to one.


In [ ]:
def verify_live_prediction_inputs():
    # File hashes do not freeze Python lists. Reload the sealed identities and
    # compare every live subset in its recorded order before using predictions.
    sealed_manifest = read_json(TRAINING_RUN / 'input_manifest.json')
    if sha256_file(TRAINING_RUN / 'input_manifest.json') != complete['inputManifestSha256']:
        raise RuntimeError('The sealed training split manifest changed.')
    sealed_inputs = read_json(RUN_DIR / 'evaluation_inputs.json')
    if sha256_file(RUN_DIR / 'evaluation_inputs.json') != INPUT_HASH:
        raise RuntimeError('Frozen evaluation inputs changed.')
    if manifest != sealed_manifest or mango_records != sealed_manifest['images']:
        raise RuntimeError('Live mango identities/order differ from the sealed training manifest.')
    if CLASS_NAMES != sealed_manifest['classNames']:
        raise RuntimeError('Live class order differs from the sealed training manifest.')
    if evaluation_inputs != sealed_inputs or assigned_records != sealed_inputs['oodImages']:
        raise RuntimeError('Live OOD assignments differ from the sealed evaluation inputs.')
    expected_subsets = [
        ('validation', validation_records, [row for row in sealed_manifest['images'] if row['subset'] == 'val']),
        ('test', test_records, [row for row in sealed_manifest['images'] if row['subset'] == 'test']),
        ('OOD tuning', tuning_records, [row for row in sealed_inputs['oodImages'] if row['subset'] == 'ood_tune']),
        ('OOD reporting', reporting_records, [row for row in sealed_inputs['oodImages'] if row['subset'] == 'ood_report']),
    ]
    for name, live_rows, sealed_rows in expected_subsets:
        if live_rows != sealed_rows:
            raise RuntimeError('Live ' + name + ' identities/order/metadata differ from the frozen subset.')
    sealed_ood = read_json(RUN_DIR / 'ood_inventory.json')['images']
    if ood_records != sealed_ood:
        raise RuntimeError('Live OOD inventory differs from the reviewed photo identities.')


def verify_frozen_inputs():
    verify_live_prediction_inputs()
    verify_artifacts()
    if sha256_file(RUN_DIR / 'evaluation_inputs.json') != INPUT_HASH:
        raise RuntimeError('Frozen evaluation inputs changed.')
    for filename, field in [
        ('ood_inventory.json', 'oodInventorySha256'),
        ('ood_review.csv', 'photoReviewSha256'),
        ('similarity_review.csv', 'similarityReviewSha256'),
    ]:
        if sha256_file(RUN_DIR / filename) != evaluation_inputs[field]:
            raise RuntimeError('A review or inventory changed after the split was frozen.')
    current_paths = {path.relative_to(OOD_ROOT).as_posix() for path in OOD_ROOT.rglob('*')
                     if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS}
    if current_paths != {record['relativePath'] for record in ood_records}:
        raise RuntimeError('The OOD folder membership changed after freezing. Start a fresh run.')
    for record in ood_records:
        verify_photo(record, OOD_ROOT)
    for record in mango_records:
        verify_photo(record, SPLIT_DIR)


def has_webp_content(path):
    '''Read the file signature because a WebP photo may have a .jpg name.'''
    with Path(path).open('rb') as source:
        header = source.read(12)
    return header[:4] == b'RIFF' and header[8:12] == b'WEBP'


def load_pixels(path):
    # Match the training loader. MobileNet rescales internally; keep 0–255 input.
    if has_webp_content(path):
        with Image.open(path) as picture:
            pixels = np.asarray(ImageOps.exif_transpose(picture).convert('RGB'), dtype=np.float32)
    else:
        pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
        pixels.set_shape((None, None, 3))
    return tf.image.resize(pixels, IMAGE_SIZE, method='bilinear', antialias=False)


def probability_stats(probabilities):
    result = []
    for row in probabilities:
        values = [float(value) for value in row]
        if len(values) != len(CLASS_NAMES) or any(not math.isfinite(value) or value < 0 or value > 1 for value in values):
            raise ValueError('Expected seven finite softmax probabilities in [0, 1].')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-4):
            raise ValueError('Probabilities must sum to one. Do not pass logits.')
        ordered = sorted(values, reverse=True)
        result.append({
            'confidence': ordered[0], 'margin': ordered[0] - ordered[1],
            'entropy': -sum(value * math.log(value) for value in values if value > 0) / math.log(len(values)),
            'predictedIndex': values.index(ordered[0]),
            **{'probability_' + label: value for label, value in zip(CLASS_NAMES, values)},
        })
    return result


def weight_digest(model):
    digest = hashlib.sha256()
    for variable in model.weights:
        values = variable.numpy()
        digest.update(str(values.shape).encode())
        digest.update(str(values.dtype).encode())
        digest.update(values.tobytes())
    return digest.hexdigest()


def predict_records(model, records, root):
    verify_live_prediction_inputs()
    allowed_groups = [
        (validation_records, SPLIT_DIR), (test_records, SPLIT_DIR),
        (tuning_records, OOD_ROOT), (reporting_records, OOD_ROOT),
    ]
    if not any(records == rows and Path(root).resolve() == Path(folder).resolve()
               for rows, folder in allowed_groups):
        raise RuntimeError('Prediction arguments do not match one complete frozen subset and its photo root.')
    predictions = []
    for start in range(0, len(records), BATCH_SIZE):
        batch_records = records[start:start + BATCH_SIZE]
        pixels = tf.stack([load_pixels(verify_photo(record, root)) for record in batch_records])
        values = model(pixels, training=False).numpy()
        if values.shape != (len(batch_records), len(CLASS_NAMES)):
            raise ValueError('The model did not return one seven-class prediction per image.')
        probability_stats(values)
        predictions.extend(values.tolist())
    if not predictions:
        raise ValueError('Cannot predict an empty image group.')
    return np.asarray(predictions, dtype=np.float32)


def accepted(stat, thresholds):
    return (
        stat['confidence'] >= thresholds['confidenceThreshold']
        and stat['margin'] >= thresholds['marginThreshold']
        and stat['entropy'] <= thresholds['entropyThreshold']
    )


def select_thresholds(validation_stats, tuning_stats, grid, confidence, minimum_retention):
    if not validation_stats or not tuning_stats or not 0 < minimum_retention <= 1:
        raise ValueError('Nonempty tuning groups and a valid retention target are required.')
    if not grid or not 0 <= confidence <= 1 or any(not math.isfinite(value) or not 0 <= value <= 1 for value in grid):
        raise ValueError('Threshold candidates must be finite numbers from zero to one.')
    required = math.ceil(minimum_retention * len(validation_stats) - 1e-9)
    best = None
    best_key = None
    search = []
    for margin in grid:
        validation_eligible = [row for row in validation_stats if row['confidence'] >= confidence and row['margin'] >= margin]
        tuning_eligible = [row for row in tuning_stats if row['confidence'] >= confidence and row['margin'] >= margin]
        for entropy in grid:
            retained = sum(row['entropy'] <= entropy for row in validation_eligible)
            rejected = len(tuning_stats) - sum(row['entropy'] <= entropy for row in tuning_eligible)
            feasible = retained >= required
            search.append({
                'margin': margin, 'entropy': entropy,
                'validationAccepted': retained, 'validationTotal': len(validation_stats),
                'validationRetention': retained / len(validation_stats),
                'oodRejected': rejected, 'oodTotal': len(tuning_stats),
                'oodRejectionRate': rejected / len(tuning_stats), 'meetsRetentionTarget': feasible,
            })
            # Exact counts decide ties. First reject more OOD; then retain more mango.
            # Further ties prefer a lower margin, then a higher maximum entropy.
            key = (rejected, retained, -margin, entropy)
            if feasible and (best_key is None or key > best_key):
                best_key = key
                best = {'confidenceThreshold': confidence, 'marginThreshold': margin, 'entropyThreshold': entropy}
    return best, search


def scored_records(records, statistics, thresholds):
    if len(records) != len(statistics):
        raise ValueError('Every photo must have exactly one prediction.')
    output = []
    for record, stat in zip(records, statistics):
        kept = accepted(stat, thresholds)
        predicted = CLASS_NAMES[stat['predictedIndex']]
        reasons = []
        if stat['confidence'] < thresholds['confidenceThreshold']:
            reasons.append('confidence')
        if stat['margin'] < thresholds['marginThreshold']:
            reasons.append('margin')
        if stat['entropy'] > thresholds['entropyThreshold']:
            reasons.append('entropy')
        output.append({
            'id': record['id'], 'relativePath': record['relativePath'], 'sha256': record['sha256'],
            'subset': record['subset'], 'actualClass': record.get('label', 'unsupported'),
            'rawPredictedClass': predicted, 'accepted': kept,
            'displayedClass': predicted if kept else 'rejected',
            'rejectionReasons': ';'.join(reasons), **stat,
        })
    return output


## Step 8: Fit gates on validation and OOD tuning only, then lock them

**What:** Sweep margin and entropy while keeping confidence at 0.70.

**Why:** Tuning against reporting or mango test photos would turn the report into another training decision.

**How:** Reproduce saved validation predictions, require at least 95% mango validation retention, maximize OOD tuning rejection, and save settings and lock hashes before reporting.

**Expected result:** A feasible selection is locked. If no gate meets the retention target, an infeasible record is saved and this run stops without test/report predictions or a candidate package.


**Tie-break order:** reject more OOD tuning photos; then retain more mango validation photos;
then choose the lower margin; then the higher maximum entropy. Counts decide ties before display rounding.
Retention means accepted true mango photos, including accepted wrong-class predictions; it is not accuracy.
This is an experimental fixed recipe, not a guarantee that unknown objects will always be rejected.


In [ ]:
SELECTION_PATH = RUN_DIR / 'selected_thresholds.json'
if SELECTION_PATH.exists() or (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('Selection already ran. Preserve this folder and start a new run to change the recipe.')
verify_frozen_inputs()

model = tf.keras.models.load_model(TRAINING_RUN / 'model_calibrated.keras', compile=False, safe_mode=True)
MODEL_WEIGHT_DIGEST = weight_digest(model)
validation_probabilities = predict_records(model, validation_records, SPLIT_DIR)
saved_validation = np.load(TRAINING_RUN / 'validation_probabilities.npy', allow_pickle=False)
if saved_validation.shape != validation_probabilities.shape:
    raise ValueError('Saved validation count/order differs from the fresh split.')
np.testing.assert_allclose(validation_probabilities, saved_validation, rtol=1e-4, atol=1e-4,
    err_msg='Validation parity failed. Check model identity and preprocessing before fitting.')
tuning_probabilities = predict_records(model, tuning_records, OOD_ROOT)
if weight_digest(model) != MODEL_WEIGHT_DIGEST:
    raise RuntimeError('Inference unexpectedly changed the model weights.')

validation_stats = probability_stats(validation_probabilities)
tuning_stats = probability_stats(tuning_probabilities)
chosen, search_rows = select_thresholds(
    validation_stats, tuning_stats, GRID,
    CONFIDENCE_THRESHOLD, MINIMUM_VALIDATION_RETENTION,
)
write_csv_once(RUN_DIR / 'threshold_search.csv', search_rows)
selection = {
    'experimentVersion': EXPERIMENT_VERSION, 'status': 'feasible' if chosen else 'infeasible',
    'classNames': CLASS_NAMES, 'thresholds': chosen,
    'trainingCompleteSha256': COMPLETE_HASH, 'evaluationInputsSha256': INPUT_HASH,
    'modelSha256': sha256_file(TRAINING_RUN / 'model_calibrated.keras'),
    'modelWeightDigest': MODEL_WEIGHT_DIGEST,
    'temperature': complete['calibratedTemperature'], 'temperatureAlreadyFolded': True,
    'confidenceFixed': CONFIDENCE_THRESHOLD, 'minimumValidationRetention': MINIMUM_VALIDATION_RETENTION,
    'gridStep': 0.01, 'searchRows': len(search_rows),
    'tieBreakOrder': ['more OOD rejected', 'more mango validation accepted', 'lower margin', 'higher maximum entropy'],
    'retentionIncludesIncorrectPredictions': True,
    'reportingUsedForFitting': False, 'testUsedForFitting': False,
    'reuseStatus': evaluation_inputs['reuseStatus'], 'frozenAt': utc_now(),
}
write_json_once(SELECTION_PATH, selection)
SELECTION_HASH = sha256_file(SELECTION_PATH)
write_json_once(RUN_DIR / 'selection_lock.json', {
    'selectionSha256': SELECTION_HASH, 'evaluationInputsSha256': INPUT_HASH,
    'trainingCompleteSha256': COMPLETE_HASH, 'frozenAt': utc_now(),
})
if chosen is None:
    raise RuntimeError('No feasible thresholds met the fixed validation retention target. Infeasibility was recorded; no reporting inference or candidate export will run.')

write_csv_once(RUN_DIR / 'validation_predictions.csv', scored_records(validation_records, validation_stats, chosen))
write_csv_once(RUN_DIR / 'ood_tuning_predictions.csv', scored_records(tuning_records, tuning_stats, chosen))
print('Frozen settings:', chosen)
print('All selection data is saved. Mango test and OOD reporting photos remain unused.')


## Step 9: Define honest classification and rejection reporting

**What:** Compute precision, recall, F1, raw accuracy, accepted-only accuracy, and rejected counts.

**Why:** Dropping rejected mango photos from every denominator can make results look better while hiding missed diagnoses.

**How:** Keep all true mango photos in class support. Count rejected photos as missed class predictions in rejection-aware recall. Report OOD false accepts separately.

**Expected result:** Raw and gated results remain separate. Accepted-only accuracy is labelled with its smaller denominator, and empty accepted sets return null rather than a made-up percentage.


In [ ]:
def classification_metrics(actual, predicted):
    if len(actual) != len(predicted):
        raise ValueError('Every true class must have exactly one predicted result.')
    if any(label not in CLASS_NAMES for label in actual):
        raise ValueError('Actual mango labels must belong to the seven supported classes.')
    if any(label not in CLASS_NAMES + ['rejected'] for label in predicted):
        raise ValueError('Predicted results must be supported classes or the rejection result.')
    metrics = []
    for label in CLASS_NAMES:
        true_positive = sum(truth == label and guess == label for truth, guess in zip(actual, predicted))
        false_positive = sum(truth != label and guess == label for truth, guess in zip(actual, predicted))
        false_negative = sum(truth == label and guess != label for truth, guess in zip(actual, predicted))
        support = sum(truth == label for truth in actual)
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        metrics.append({
            'className': label, 'precision': precision, 'recall': recall, 'f1Score': f1,
            'support': support, 'truePositive': true_positive,
            'falsePositive': false_positive, 'falseNegative': false_negative,
        })
    return metrics


def summarize_mango(scored):
    if not scored:
        raise ValueError('A mango reporting subset cannot be empty.')
    actual = [row['actualClass'] for row in scored]
    raw_predicted = [row['rawPredictedClass'] for row in scored]
    selected_predicted = [row['displayedClass'] for row in scored]
    accepted_count = sum(row['accepted'] for row in scored)
    correct_raw = sum(truth == predicted for truth, predicted in zip(actual, raw_predicted))
    correct_accepted = sum(row['accepted'] and row['actualClass'] == row['rawPredictedClass'] for row in scored)
    raw_matrix = [[sum(truth == first and predicted == second for truth, predicted in zip(actual, raw_predicted)) for second in CLASS_NAMES] for first in CLASS_NAMES]
    rejection_matrix = [[sum(truth == first and predicted == second for truth, predicted in zip(actual, selected_predicted)) for second in CLASS_NAMES + ['rejected']] for first in CLASS_NAMES]
    return {
        'total': len(scored), 'accepted': accepted_count, 'rejected': len(scored) - accepted_count,
        'retention': accepted_count / len(scored), 'rawAccuracy': correct_raw / len(scored),
        'correctAcceptedFractionOfAllPhotos': correct_accepted / len(scored),
        'acceptedOnlyAccuracy': correct_accepted / accepted_count if accepted_count else None,
        'rawPerClassMetrics': classification_metrics(actual, raw_predicted),
        'rejectionAwarePerClassMetrics': classification_metrics(actual, selected_predicted),
        'rawConfusionMatrix': raw_matrix, 'rejectionAwareConfusionMatrix': rejection_matrix,
        'matrixRowLabels': CLASS_NAMES, 'rejectionMatrixColumnLabels': CLASS_NAMES + ['rejected'],
        'denominatorNote': 'All true mango photos remain in support. Rejected true mango photos count as missed class predictions.',
    }


def summarize_ood(scored):
    if not scored:
        raise ValueError('An OOD reporting subset cannot be empty.')
    false_accepts = sum(row['accepted'] for row in scored)
    return {
        'total': len(scored), 'falseAccepted': false_accepts, 'rejected': len(scored) - false_accepts,
        'falseAcceptRate': false_accepts / len(scored),
        'rejectionRate': (len(scored) - false_accepts) / len(scored),
    }


def read_selection_lock():
    frozen = read_json(SELECTION_PATH)
    lock = read_json(RUN_DIR / 'selection_lock.json')
    if sha256_file(SELECTION_PATH) != lock['selectionSha256'] or lock['selectionSha256'] != SELECTION_HASH:
        raise RuntimeError('The frozen rejection selection changed.')
    if frozen['status'] != 'feasible' or frozen['evaluationInputsSha256'] != INPUT_HASH:
        raise RuntimeError('No valid frozen gate belongs to these audited inputs.')
    if frozen['trainingCompleteSha256'] != COMPLETE_HASH or frozen['modelSha256'] != sha256_file(TRAINING_RUN / 'model_calibrated.keras'):
        raise RuntimeError('The frozen gate belongs to another model.')
    return frozen


## Step 10: Run the locked mango test and OOD reporting evaluation once

**What:** Predict the reporting subsets only after the immutable selection exists.

**Why:** A report should describe the chosen recipe, rather than a recipe changed after seeing its weaknesses.

**How:** Write an evaluation-start marker, verify model/input identities, evaluate frozen settings and a confidence-only reference using the same predictions, then seal all measurements.

**Expected result:** CSV predictions, per-class metrics, raw/gated matrices, and evaluation hashes are saved. Interrupted reporting requires a fresh run; this run cannot be silently retuned.


In [ ]:
if (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('Evaluation already started in this folder. Preserve its evidence; do not retune or rerun it.')
frozen = read_selection_lock()
verify_frozen_inputs()
if weight_digest(model) != frozen['modelWeightDigest']:
    raise RuntimeError('In-memory model weights differ from the selected model.')
write_json_once(RUN_DIR / 'evaluation_started.json', {
    'selectionSha256': SELECTION_HASH, 'evaluationInputsSha256': INPUT_HASH,
    'trainingCompleteSha256': COMPLETE_HASH, 'startedAt': utc_now(),
})

test_probabilities = predict_records(model, test_records, SPLIT_DIR)
report_probabilities = predict_records(model, reporting_records, OOD_ROOT)
if weight_digest(model) != frozen['modelWeightDigest']:
    raise RuntimeError('Reporting inference changed the model weights.')

test_stats = probability_stats(test_probabilities)
report_stats = probability_stats(report_probabilities)
selected_test_rows = scored_records(test_records, test_stats, frozen['thresholds'])
selected_report_rows = scored_records(reporting_records, report_stats, frozen['thresholds'])
confidence_only = {'confidenceThreshold': frozen['confidenceFixed'], 'marginThreshold': 0.0, 'entropyThreshold': 1.0}
reference_test_rows = scored_records(test_records, test_stats, confidence_only)
reference_report_rows = scored_records(reporting_records, report_stats, confidence_only)

write_csv_once(RUN_DIR / 'mango_test_predictions.csv', selected_test_rows)
write_csv_once(RUN_DIR / 'ood_report_predictions.csv', selected_report_rows)
write_csv_once(RUN_DIR / 'confidence_only_mango_test_predictions.csv', reference_test_rows)
write_csv_once(RUN_DIR / 'confidence_only_ood_report_predictions.csv', reference_report_rows)

evaluation = {
    'experimentVersion': EXPERIMENT_VERSION, 'status': 'experimental; not adopted or deployed',
    'classNames': CLASS_NAMES, 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_HASH, 'selectionSha256': SELECTION_HASH,
    'evaluationInputsSha256': INPUT_HASH, 'thresholds': frozen['thresholds'],
    'mangoValidation': summarize_mango(scored_records(validation_records, validation_stats, frozen['thresholds'])),
    'oodTuning': summarize_ood(scored_records(tuning_records, tuning_stats, frozen['thresholds'])),
    'mangoTest': summarize_mango(selected_test_rows),
    'oodReport': summarize_ood(selected_report_rows),
    'confidenceOnlyReference': {
        'mangoTest': summarize_mango(reference_test_rows), 'oodReport': summarize_ood(reference_report_rows),
    },
    'reuseStatus': evaluation_inputs['reuseStatus'],
    'supportedDatasetStatus': 'Fresh grouped split of existing study mango photographs; not independently collected field accuracy.',
    'browserPredictionParityMeasured': False,
    'finishedAt': utc_now(),
}
write_csv_once(RUN_DIR / 'raw_per_class_metrics.csv', evaluation['mangoTest']['rawPerClassMetrics'])
write_csv_once(RUN_DIR / 'rejection_aware_per_class_metrics.csv', evaluation['mangoTest']['rejectionAwarePerClassMetrics'])
# Photo history categories describe reviewed provenance, not guaranteed independent collection.
report_by_id = {row['id']: row for row in selected_report_rows}
tune_by_id = {row['id']: row for row in scored_records(tuning_records, tuning_stats, frozen['thresholds'])}
evaluation['oodPriorUseBreakdown'] = {}
for subset, records, predictions in [
    ('ood_tune', tuning_records, tune_by_id), ('ood_report', reporting_records, report_by_id),
]:
    evaluation['oodPriorUseBreakdown'][subset] = {
        history: summarize_ood([predictions[row['id']] for row in records if row['priorUse'] == history])
        for history in ['previously_evaluated', 'new_for_v4_5_0', 'unknown']
        if any(row['priorUse'] == history for row in records)
    }
write_json_once(RUN_DIR / 'evaluation.json', evaluation)
EVALUATION_HASH = sha256_file(RUN_DIR / 'evaluation.json')
write_json_once(RUN_DIR / 'evaluation_lock.json', {
    'evaluationSha256': EVALUATION_HASH, 'selectionSha256': SELECTION_HASH,
    'evaluationInputsSha256': INPUT_HASH, 'finishedAt': utc_now(),
})

print('Raw mango test accuracy:', evaluation['mangoTest']['rawAccuracy'])
print('Mango acceptance fraction:', evaluation['mangoTest']['retention'])
print('Correct accepted / ALL mango photos:', evaluation['mangoTest']['correctAcceptedFractionOfAllPhotos'])
print('Accuracy among accepted mango photos ONLY:', evaluation['mangoTest']['acceptedOnlyAccuracy'])
print('Reused OOD reporting results:', evaluation['oodReport'])


## Step 11: Draw V4.5.0 figures from the new measurements

**What:** Create confusion matrices, grouped class-metric charts, and a threshold-search plot.

**Why:** The familiar V2.1 layout helps comparison, but its historical numbers cannot describe this new model.

**How:** Use only this run's locked evaluation and search rows. Counts and row percentages are shown side by side; each class metric includes its true test support. Captions appear underneath each figure.

**Expected result:** Figures are labelled V4.5.0 and use actual new counts/scores. Raw figures and rejection-aware figures clearly describe their different denominators.


In [ ]:
frozen = read_selection_lock()
if sha256_file(RUN_DIR / 'evaluation.json') != EVALUATION_HASH:
    raise RuntimeError('Reporting measurements changed before plotting.')
labels = [label.replace('_', ' ').title() for label in CLASS_NAMES]
captions = {}

def finish_figure(figure, filename, caption):
    figure.tight_layout(rect=(0, 0.04, 1, 0.96))
    figure.text(0.5, 0.01, caption, ha='center', va='bottom', fontsize=9, wrap=True)
    figure.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    captions[filename] = caption
    display(Markdown('**' + filename + ':** ' + caption))

def plot_matrix_pair(matrix, column_labels, title, filename, caption):
    values = np.asarray(matrix, dtype=int)
    row_totals = values.sum(axis=1, keepdims=True)
    percentages = np.divide(values * 100.0, row_totals,
                            out=np.zeros_like(values, dtype=float), where=row_totals != 0)
    figure, axes = plt.subplots(1, 2, figsize=(22, 9))
    for axis, panel, panel_title, as_percent in [
        (axes[0], values, 'Counts', False),
        (axes[1], percentages, 'Row percentages (recall on diagonal)', True),
    ]:
        image = axis.imshow(panel, cmap='Blues', vmin=0, vmax=100 if as_percent else None)
        figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
        contrast = 50 if as_percent else max(1, float(panel.max()) / 2)
        for row in range(panel.shape[0]):
            for column in range(panel.shape[1]):
                value = panel[row, column]
                text = f'{value:.1f}%' if as_percent else str(int(value))
                axis.text(column, row, text, ha='center', va='center', fontsize=9,
                          color='white' if value > contrast else 'black')
        axis.set_xticks(range(len(column_labels)), column_labels, rotation=45, ha='right')
        axis.set_yticks(range(len(labels)), labels)
        axis.set_xlabel('Predicted result')
        axis.set_ylabel('Actual class')
        axis.set_title(panel_title)
    figure.suptitle(title, fontsize=16)
    finish_figure(figure, filename, caption)

mango = evaluation['mangoTest']
plot_matrix_pair(
    mango['rawConfusionMatrix'], labels,
    f"V4.5.0 confusion matrix on held-out mango test (n = {mango['total']})",
    'confusion_matrix.png',
    'Raw seven-class predictions before rejection. Rows are actual classes; columns are predicted classes. '
    'Each right-panel row sums to 100%; its diagonal is class recall. This is existing-study test evidence.',
)
plot_matrix_pair(
    mango['rejectionAwareConfusionMatrix'], labels + ['Rejected'],
    f"V4.5.0 confusion matrix with frozen rejection gates (n = {mango['total']})",
    'confusion_matrix_with_rejection.png',
    'All true mango test photos remain in the denominator. The Rejected column shows missed diagnoses; '
    'diagonal row percentages are rejection-aware recall. Rejection is a gate, not an eighth trained class.',
)

def plot_class_metrics(rows, overall_score, overall_label, title, filename, caption):
    locations = np.arange(len(rows))
    figure, axis = plt.subplots(figsize=(17, 8))
    for offset, field, color, label in [
        (-0.26, 'precision', '#1f77b4', 'Precision'),
        (0.00, 'recall', '#ff7f0e', 'Recall'),
        (0.26, 'f1Score', '#2ca02c', 'F1-score'),
    ]:
        values = [row[field] for row in rows]
        bars = axis.bar(locations + offset, values, width=0.25, color=color, label=label)
        axis.bar_label(bars, labels=[f'{value:.2f}' for value in values], padding=3, fontsize=9)
    axis.axhline(overall_score, color='gray', linestyle='--',
                 label=f'{overall_label} {100 * overall_score:.2f}%')
    axis.set_xticks(locations, [label + '\n(n=' + str(row['support']) + ')'
                               for label, row in zip(labels, rows)], rotation=15, ha='right')
    axis.set_ylim(0, 1.12)
    axis.set_ylabel('Score (1.0 = 100%)')
    axis.set_title(title)
    axis.grid(axis='y', alpha=0.2)
    axis.legend(loc='lower right')
    finish_figure(figure, filename, caption)

plot_class_metrics(
    mango['rawPerClassMetrics'], mango['rawAccuracy'], 'Overall raw accuracy',
    'V4.5.0 per-class precision, recall, and F1 on held-out mango test',
    'per_class_metrics.png',
    'Blue precision: how often a predicted class is correct. Orange recall: how many real class photos were found. '
    'Green F1 balances both. Support n is the actual class count; bar labels are rounded for display only.',
)
plot_class_metrics(
    mango['rejectionAwarePerClassMetrics'], mango['correctAcceptedFractionOfAllPhotos'],
    'Correct accepted / all mango photos',
    'V4.5.0 per-class metrics with frozen rejection gates',
    'per_class_metrics_with_rejection.png',
    'All true mango test photos remain in class support. Rejected mango photos count as missed class predictions. '
    'The dashed line is correct accepted predictions divided by all test photos, not accepted-only accuracy.',
)

selected_margin = frozen['thresholds']['marginThreshold']
selected_entropy = frozen['thresholds']['entropyThreshold']
slice_rows = [row for row in search_rows if row['margin'] == selected_margin]
figure, axis = plt.subplots(figsize=(12, 7))
axis.plot([row['entropy'] for row in slice_rows],
          [100 * row['validationRetention'] for row in slice_rows], label='Mango validation accepted')
axis.plot([row['entropy'] for row in slice_rows],
          [100 * row['oodRejectionRate'] for row in slice_rows], label='OOD tuning rejected')
axis.axhline(100 * MINIMUM_VALIDATION_RETENTION, color='gray', linestyle='--', label='95% mango retention target')
axis.axvline(selected_entropy, color='crimson', linestyle=':', label='Frozen maximum entropy')
axis.set_title('V4.5.0 threshold search using validation and OOD tuning only')
axis.set_xlabel(f'Maximum normalized entropy; margin = {selected_margin:.2f}, confidence = 0.70')
axis.set_ylabel('Photos (%)')
axis.set_ylim(0, 103)
axis.grid(alpha=0.2)
axis.legend()
finish_figure(
    figure, 'threshold_search.png',
    'One slice of the two-dimensional margin/entropy search. The frozen choice must retain at least 95% '
    'of mango validation photos. OOD reporting and mango test scores were never used to choose this point.',
)
write_json_once(FIGURE_DIR / 'figure_captions.json', captions)


## Step 12: Package the sealed browser model with frozen settings

**What:** Copy the training export and attach the locked rejection gates.

**Why:** Changing graph or weight bytes after evaluation would create a different candidate from the measured model.

**How:** Verify all TensorFlow.js shard identities, copy identical graph/shards, write candidate configuration, and make a ZIP with SHA-256 records.

**Expected result:** candidate_model.zip contains the new seven-class model and chosen gates. The current PWA model remains unchanged, and browser parity remains pending until it is measured.


In [ ]:
read_selection_lock()
verify_frozen_inputs()
evaluation_lock = read_json(RUN_DIR / 'evaluation_lock.json')
if sha256_file(RUN_DIR / 'evaluation.json') != evaluation_lock['evaluationSha256']:
    raise RuntimeError('Evaluation records changed before packaging.')
if evaluation_lock['selectionSha256'] != SELECTION_HASH or evaluation_lock['evaluationInputsSha256'] != INPUT_HASH:
    raise RuntimeError('Evaluation does not belong to the frozen gate and inputs.')

source_export = TRAINING_RUN / 'tfjs_model'
browser_json = read_json(source_export / 'model.json')
required_browser_files = {'model.json', 'model_config.json'}
for group in browser_json['weightsManifest']:
    for relative in group['paths']:
        inside(source_export, relative)
        required_browser_files.add(relative)
if len(required_browser_files) < 3:
    raise ValueError('The browser export has no weight shards.')

candidate_root = RUN_DIR / 'candidate_model'
candidate_root.mkdir(exist_ok=False)
for relative in sorted(required_browser_files):
    expected_key = 'tfjs_model/' + relative
    if expected_key not in complete['artifacts']:
        raise ValueError('Browser file is absent from the completed training manifest: ' + relative)
    source = inside(source_export, relative)
    target = inside(candidate_root, relative)
    target.parent.mkdir(parents=True, exist_ok=True)
    if relative != 'model_config.json':
        shutil.copyfile(source, target)
        if sha256_file(target) != complete['artifacts'][expected_key]['sha256']:
            raise RuntimeError('A copied browser model file differs from the fresh training export.')

candidate_config = read_json(source_export / 'model_config.json')
if candidate_config.get('classes', candidate_config.get('classNames')) != CLASS_NAMES:
    raise ValueError('Browser configuration class order differs from the calibrated model.')
if candidate_config.get('temperatureFoldedIntoWeights') is not True:
    raise ValueError('Browser export must contain the already calibrated model.')
if not math.isclose(float(candidate_config['temperature']), float(complete['calibratedTemperature']), rel_tol=1e-12):
    raise ValueError('Browser export temperature metadata differs from the completed run.')
candidate_config.update({
    'modelVersion': 'model-v4.5.0-candidate', 'experimentVersion': EXPERIMENT_VERSION,
    'thresholds': frozen['thresholds'], 'status': 'experimental candidate; not adopted or deployed',
    'selectionSha256': SELECTION_HASH, 'evaluationSha256': EVALUATION_HASH,
    'trainingCompleteSha256': COMPLETE_HASH, 'browserPredictionParityMeasured': False,
})
write_json_once(candidate_root / 'model_config.json', candidate_config)
candidate_hashes = {
    path.relative_to(candidate_root).as_posix(): {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
    for path in sorted(candidate_root.rglob('*')) if path.is_file()
}
write_json_once(RUN_DIR / 'candidate_hashes.json', candidate_hashes)

with zipfile.ZipFile(RUN_DIR / 'candidate_model.zip', 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for relative in sorted(candidate_hashes):
        archive.write(inside(candidate_root, relative), arcname=relative)
print('Experimental browser export:', candidate_root)
print('Application model adoption requires a separate decision and applicable evaluation.')


## Step 13: Seal the threshold experiment and read the result limits

**What:** Write a completion receipt for the explicit evidence files this workflow created.

**Why:** Native Google Sheets pointers and unrelated uploads are not readable scientific artifacts and must not enter a recursive file hash.

**How:** Hash only known JSON/CSV/ZIP/image files, verify selection/evaluation/input locks again, and save the final completion record once.

**Expected result:** The completed threshold run is traceable to its fresh training model, reviewed source inventories, locked gates, and actual evaluation. No deployment or field accuracy is claimed.


In [ ]:
read_selection_lock()
verify_frozen_inputs()
if sha256_file(RUN_DIR / 'evaluation.json') != EVALUATION_HASH:
    raise RuntimeError('Final reporting evidence changed.')
evidence_files = [
    'runtime.json', 'training_reference.json', 'ood_inventory.json', 'ood_review.csv',
    'similarity_review.csv', 'evaluation_inputs.json', 'ood_assignments.csv', 'input_lock.json',
    'threshold_search.csv', 'selected_thresholds.json', 'selection_lock.json',
    'validation_predictions.csv', 'ood_tuning_predictions.csv', 'evaluation_started.json',
    'mango_test_predictions.csv', 'ood_report_predictions.csv',
    'confidence_only_mango_test_predictions.csv', 'confidence_only_ood_report_predictions.csv',
    'raw_per_class_metrics.csv', 'rejection_aware_per_class_metrics.csv',
    'evaluation.json', 'evaluation_lock.json', 'candidate_hashes.json', 'candidate_model.zip',
    'figures/confusion_matrix.png', 'figures/confusion_matrix_with_rejection.png',
    'figures/per_class_metrics.png', 'figures/per_class_metrics_with_rejection.png',
    'figures/threshold_search.png', 'figures/figure_captions.json',
]
evidence_files += ['candidate_model/' + relative for relative in candidate_hashes]
evidence_files += [path.relative_to(RUN_DIR).as_posix()
                   for path in sorted((RUN_DIR / 'similarity_contact_sheets').glob('pairs_*.jpg'))]
evidence_files += [path.relative_to(RUN_DIR).as_posix()
                   for path in sorted((RUN_DIR / 'review_contact_sheets').glob('ood_*.jpg'))]
evidence_files += [path.relative_to(RUN_DIR).as_posix()
                   for path in sorted((RUN_DIR / 'runtime_preflight').iterdir()) if path.is_file()]
artifacts = {
    relative: {'sha256': sha256_file(inside(RUN_DIR, relative)),
               'bytes': inside(RUN_DIR, relative).stat().st_size}
    for relative in sorted(set(evidence_files))
}
write_json_once(RUN_DIR / 'threshold_fitting_complete.json', {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION,
    'status': 'completed threshold experiment; not adopted or deployed',
    'trainingRun': str(TRAINING_RUN), 'trainingCompleteSha256': COMPLETE_HASH,
    'selectionSha256': SELECTION_HASH, 'evaluationSha256': EVALUATION_HASH,
    'evaluationInputsSha256': INPUT_HASH, 'classNames': CLASS_NAMES,
    'thresholds': frozen['thresholds'], 'oodCounts': dict(assignment_counts),
    'modelWeightsChangedByThisNotebook': False,
    'reuseStatus': evaluation_inputs['reuseStatus'],
    'browserPredictionParityMeasured': False,
    'artifacts': artifacts, 'finishedAt': utc_now(),
})
print('Completed experiment:', RUN_DIR)
print('Save the executed notebook with this run. Its original template contains no claimed results.')


## How to understand the completed outputs

- **Raw mango accuracy:** correct top-class guesses divided by every mango test photo.
- **Retention:** accepted true mango photos divided by every mango test photo; accepted guesses can still be wrong.
- **Correct accepted / all mango photos:** useful correct predictions after rejection, with all test photos in support.
- **Accepted-only accuracy:** correctness among accepted mango photos only; always read it together with retention.
- **OOD false accept:** unsupported content that still passes the gates and receives one of the seven mango labels.
- **Raw confusion matrix:** actual classes are rows, guesses are columns, and the diagonal counts correct predictions.
  Its right panel divides each row by that class's support, so diagonal percentages are recall.
- **OE:** auxiliary training photos encourage uncertainty on unsupported content. OE is not an eighth class and
  does not guarantee rejection of every unseen object. These reporting photos must not have trained that OE loss.

A new grouped split of existing mango study photos is not independently collected phone/field evidence.
Reused OOD material and exact photo histories stay visible in the saved CSVs. Different augmentations plus OE
change several factors together; their combined outcome does not isolate the effect of OE alone.
This notebook never updates `public/model/`, changes the shipped application version, or promotes a model.

Save the executed notebook alongside this run's completion receipt. Preserve an infeasible selection too;
do not lower the target after seeing reporting results and present it as the original recipe.
